# Lab 4: your first model API calls

**About 50–60 minutes · work with a partner, but use your own key**

A chat interface hides most of the machinery. In this lab, you will call models from Python and inspect the pieces that an application sees: a URL, a model ID, messages, a response, token counts, and vectors.

By the end, you should be able to:

1. keep an API key out of code and notebook output;
2. explain the request/response pattern of an API call;
3. choose a model by its input and output type; and
4. use an embedding to compare the meanings of short texts.

> **Separate activity:** Your instructor may also use lab time for the HW4 starter repository and `tiny_rnn.py`. Nothing in this notebook changes or depends on that repository.

## 0. Before you run anything: protect the key

An API key is a password that can spend your course credit. A usable OpenRouter key begins with `sk-or-`. A key beginning with `ytf` is for managing the Yale SOM Token Fund website and cannot call models.

**Never put a key in:**

- a Python string such as `api_key = "sk-or-..."`;
- a prompt to ChatGPT, Claude, Codex, or another assistant;
- a screenshot, discussion post, or shared document;
- a Git commit or GitHub repository; or
- notebook output.

If a key is exposed, disable it and create a replacement.

### Add the key as a Colab secret

1. In Colab, click the **key icon** in the left sidebar.
2. Add a secret named exactly `OPENROUTER_API_KEY`.
3. Paste your `sk-or-...` key as its value.
4. Turn on **Notebook access** for that secret.

The code below asks Colab for the secret but never prints it. If you are running locally, set an environment variable with the same name before starting Jupyter.

In [ ]:
%pip -q install "openai>=1.0" requests


In [ ]:
import os
import requests
import numpy as np
from openai import OpenAI

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("Add OPENROUTER_API_KEY as a Colab secret, then run again.")
if not OPENROUTER_API_KEY.startswith("sk-or-"):
    raise ValueError("This does not look like an OpenRouter key. Did you copy a ytf management key?")

BASE_URL = "https://openrouter.ai/api/v1"
HEADERS = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
client = OpenAI(base_url=BASE_URL, api_key=OPENROUTER_API_KEY)
print("Ready. The key was loaded but not displayed.")

## 1. Ask the API about the key

An API is a set of URLs that accept structured requests. Here we send a `GET` request to `/key`. The server uses the secret header to identify the key and returns JSON data about it.

In [ ]:
key_response = requests.get(f"{BASE_URL}/key", headers=HEADERS, timeout=30)
key_response.raise_for_status()
key_info = key_response.json()["data"]

print("Spending limit:   $", key_info.get("limit"), sep="")
print("Already used:     $", key_info.get("usage"), sep="")
print("Remaining:        $", key_info.get("limit_remaining"), sep="")
print("Expiration (UTC):", key_info.get("expires_at"))

**Checkpoint 1 — discuss with a partner**

- Which part of the request proves who you are?
- Which part names the resource you want?
- Why did we print selected fields instead of the entire response or the key?

Write a one-sentence answer here:

> Your answer:

## 2. Model IDs are addresses, not personalities

Your key can now call **every model on OpenRouter**: several hundred, from dozens of organizations. (When this lab was first released, only five were enabled.) A model ID such as `openai/gpt-6-luna` is an address: the organization, then the model. It tells OpenRouter where to route your request.

You do not need most of them. For everyday experiments, this cheap shortlist covers the main kinds of input and output:

| Model ID | Input | Output | Good first question |
|---|---|---|---|
| `deepseek/deepseek-v4.1-flash` | text, image | text | Can it answer a short prompt? |
| `openai/gpt-6-luna` | text, image, file | text | How does another text model respond? |
| `openai/text-embedding-3-small` | text | vector | Which sentences are similar? |
| `google/gemini-3.1-flash-lite-image` | text, image | text, image | How can a model produce an image? |
| `openai/gpt-audio-mini` | text, audio | text, audio | How does an app send and receive sound? |

We will call one text model and the embedding model today. The image and audio models need different request and display code, so we will map their capabilities without spending credit on them. An optional section near the end sends one prompt to models across the whole price range.

For help reading the full list, see this short [model overview](https://glbl5053.github.io/labs/week-05/models.html).

In [ ]:
models_response = requests.get(f"{BASE_URL}/models/user", headers=HEADERS, timeout=30)
models_response.raise_for_status()
available_model_ids = [model["id"] for model in models_response.json()["data"]]
print(len(available_model_ids), "generative models are visible to this key.")

# The public catalog: prices and capabilities for every model, no key needed.
catalog = {model["id"]: model for model in requests.get(f"{BASE_URL}/models", timeout=60).json()["data"]}

def price_per_million(model_id):
    pricing = catalog[model_id]["pricing"]
    return float(pricing["prompt"]) * 1_000_000, float(pricing["completion"]) * 1_000_000

print("\nCheap shortlist, in dollars per million tokens (input / output):")
for model_id in ["deepseek/deepseek-v4.1-flash", "openai/gpt-6-luna",
                 "google/gemini-3.1-flash-lite-image", "openai/gpt-audio-mini"]:
    if model_id in catalog:
        p_in, p_out = price_per_million(model_id)
        reachable = "" if model_id in available_model_ids else "   (not visible to this key)"
        print(f" - {model_id}: ${p_in:.2f} / ${p_out:.2f}{reachable}")
    else:
        print(f" - {model_id}: no longer in the catalog")

print("\nNote: the embedding model is tested through the embeddings endpoint in section 4.")

## 3. Send one chat request

Before running the next cell, predict the four main ingredients of the request. Then find them in the code:

1. **destination** — the client's base URL;
2. **authorization** — the API key loaded during setup;
3. **model** — the model ID; and
4. **input** — a list of messages with roles and content.

In [ ]:
prompt = "Explain a recurrent neural network in two sentences for a student who knows what a loop is."

chat_response = client.chat.completions.create(
    model="deepseek/deepseek-v4.1-flash",
    messages=[
        {"role": "system", "content": "Be concrete and concise."},
        {"role": "user", "content": prompt},
    ],
    max_tokens=300,
    extra_body={"reasoning": {"enabled": False}},
)

choice = chat_response.choices[0]
answer = choice.message.content
if answer:
    print(answer)
else:
    print("No visible answer was returned. Finish reason:", choice.finish_reason)
print("\nModel returned:", chat_response.model)
print("Prompt tokens:", chat_response.usage.prompt_tokens)
print("Completion tokens:", chat_response.usage.completion_tokens)

### Change one thing, not everything

Choose **one** experiment below. Make a prediction first, edit the next cell, then run it.

- Change the audience from “a student who knows what a loop is” to “a ten-year-old.”
- Ask for an analogy and one limitation of the analogy.
- Change the model to `openai/gpt-6-luna` while keeping the prompt identical. Any model ID now works; to see the extremes, try `anthropic/claude-sonnet-5.5` or the tiny `meta-llama/llama-3.2-1b-instruct`.
- Replace the system message with `Use exactly three bullet points.`

Do not repeatedly rerun a request just to hunt for a perfect answer. Each run uses credit.

In [ ]:
# Change ONE variable below. Write your prediction in this comment first:
# I predict ...

experiment_model = "deepseek/deepseek-v4.1-flash"
experiment_system = "Be concrete and concise."
experiment_prompt = prompt

def reasoning_extra(model_id):
    # Many models "think" before they answer, and you pay for those tokens. Switch thinking off where
    # that is allowed. A few models require it and reject a request to disable it: ask those for the
    # least effort they offer.
    info = catalog.get(model_id, {})
    if "reasoning" not in info.get("supported_parameters", []):
        return {}
    setting = info.get("reasoning") or {}
    if not setting.get("mandatory"):
        return {"reasoning": {"enabled": False}}
    for effort in ("minimal", "low", "medium", "high"):
        if effort in (setting.get("supported_efforts") or []):
            return {"reasoning": {"effort": effort}}
    return {}

experiment_response = client.chat.completions.create(
    model=experiment_model,
    messages=[
        {"role": "system", "content": experiment_system},
        {"role": "user", "content": experiment_prompt},
    ],
    max_tokens=1000,   # room for models that must think before answering
    extra_body=reasoning_extra(experiment_model),
)
experiment_choice = experiment_response.choices[0]
experiment_answer = experiment_choice.message.content
if experiment_answer:
    print(experiment_answer)
else:
    print("No visible answer was returned. Finish reason:", experiment_choice.finish_reason)

**Checkpoint 2 — record evidence, not a vibe**

Complete all three lines:

> I changed:  
> I predicted:  
> I observed (quote or describe one specific difference):

## 4. A different output: embeddings

A chat model returns words. An embedding model returns a list of numbers—a vector—that places a piece of text in a mathematical space. Texts used in similar ways tend to receive vectors that point in similar directions.

We will compare vectors with **cosine similarity**. A value near 1 means the vectors point in similar directions; a lower value means they are less similar. This is useful evidence, not a proof that two sentences mean the same thing.

In [ ]:
texts = [
    "The cat slept on the warm windowsill.",
    "A kitten napped in a sunny spot by the window.",
    "Gradient descent updates parameters to reduce a loss.",
]

embedding_response = client.embeddings.create(
    model="openai/text-embedding-3-small",
    input=texts,
)
vectors = [np.array(item.embedding) for item in embedding_response.data]

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Vector length:", len(vectors[0]))
print("sentence 1 vs 2:", round(cosine_similarity(vectors[0], vectors[1]), 3))
print("sentence 1 vs 3:", round(cosine_similarity(vectors[0], vectors[2]), 3))

### Build a three-sentence test

Replace the texts below with:

1. a sentence about a topic you choose;
2. a paraphrase that keeps the meaning but changes most of the words; and
3. a sentence on a different topic that shares at least one word with sentence 1.

Predict which pair will have the higher similarity. Then run the cell and explain whether the result supports your prediction.

In [ ]:
my_texts = [
    "Replace this with sentence 1.",
    "Replace this with a paraphrase.",
    "Replace this with a different topic sharing one word.",
]

my_response = client.embeddings.create(
    model="openai/text-embedding-3-small",
    input=my_texts,
)
my_vectors = [np.array(item.embedding) for item in my_response.data]

print("sentence 1 vs 2:", round(cosine_similarity(my_vectors[0], my_vectors[1]), 3))
print("sentence 1 vs 3:", round(cosine_similarity(my_vectors[0], my_vectors[2]), 3))

## Optional: one prompt, across the whole range

Now that every model is available, send the same prompt to five of them, from a frontier model to one small enough to run on a phone. Before you run the cell, predict: which is the cheapest model whose answer you would accept?

The five calls together should cost a few cents at most. The cell prints the actual cost of each.

In [ ]:
import time

range_models = [
    "anthropic/claude-fable-5.1",         # frontier
    "anthropic/claude-sonnet-5.5",        # workhorse
    "openai/gpt-6-luna",                  # fast and cheap
    "google/gemma-4-31b-it",              # small, open weights
    "meta-llama/llama-3.2-1b-instruct",   # tiny
]
range_prompt = prompt   # the RNN question from section 3; replace it with your own if you like

for model_id in range_models:
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": range_prompt}],
            max_tokens=1000,
            extra_body=reasoning_extra(model_id),
        )
        choice = response.choices[0]
        text = choice.message.content or f"(no visible answer; finish reason: {choice.finish_reason})"
        tokens_out = response.usage.completion_tokens
        cost = getattr(response.usage, "cost", None)
        if cost is None and model_id in catalog:   # no bill attached: estimate from list prices
            p_in, p_out = price_per_million(model_id)
            cost = (response.usage.prompt_tokens * p_in + tokens_out * p_out) / 1_000_000
    except Exception as error:
        text, tokens_out, cost = f"(request failed: {str(error)[:160]})", "-", None
    cost_text = "cost unknown" if cost is None else f"${cost:.5f}"
    print("=" * 78)
    print(f"{model_id}  |  {time.time() - start:.1f} s  |  {tokens_out} tokens out  |  {cost_text}")
    print("-" * 78)
    print(text)
print("=" * 78)

**Checkpoint — record evidence, not a vibe**

> Cheapest model whose answer I would accept:  
> One specific difference between the first and last answers (quote it):  
> Ratio of the most expensive call to the cheapest:

If a row says the request failed, note the model and the message. A small model served by a single host is more likely to be busy when a whole class calls it at once.

For a longer version of this experiment (nine models, questions with a right answer, and a blind ranking), see the [GLBL 5053 notebook](https://colab.research.google.com/github/glbl5053/labs/blob/main/week-05/openrouter-model-range.ipynb).

## 5. Exit ticket

Answer briefly in your own words.

1. What is one difference between calling a chat model and calling an embedding model?
2. Name two places where an API key should never appear.
3. In the chat request, what would you change to switch models while keeping the experiment otherwise fair?
4. What did your embedding experiment show? Include the two similarity values.
5. Check the `/key` endpoint again. How much usage is reported now, and why might a very small amount initially appear rounded?

When you are finished, use **Runtime → Disconnect and delete runtime** in Colab. The key stays in Colab's secret manager, not in the notebook. Follow your instructor's directions for what, if anything, to submit.